In [9]:
# home llms, office olm2
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning, message=".*escape sequence.*")

import httpx
from openai import OpenAI
import time
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv, find_dotenv
from statistics import mode
import ast
import re

#from aux_prompts import *
#from queries import criteria_text3
os.environ["OPENAI_API_KEY"] = 'hey' # for importing trialmind, api_key should be set beforehand
os.environ['TEMPERATURE'] = '0'

use = 'rogozin'
#use = 'cpp'

if use == 'hf':
    os.environ["BASE_URL"] = "https://router.huggingface.co/v1"
    os.environ["MODEL_NAME"] ='Qwen/Qwen3-14B'
    os.environ["OPENAI_API_KEY"] = os.getenv("HUGGINGFACE_HUB_TOKEN")
elif use == 'olm':
    os.environ["BASE_URL"] = "http://localhost:11434/v1"
    os.environ["MODEL_NAME"] ='qwen3:14b'
    os.environ["OPENAI_API_KEY"] = 'hey'
elif use == 'rogozin':
    os.environ["BASE_URL"] = "http://localhost:8501/v1"
    os.environ["MODEL_NAME"] ='/models/Qwen3.6-27B-UD-Q4_K_XL.gguf'#'ggml-org/gemma-3-1b-it-GGUF'
    os.environ["TRANSL_MODEL_NAME"] = '/models/Qwen3.6-27B-UD-Q4_K_XL.gguf'
    os.environ["OPENAI_API_KEY"] = 'hello'
elif use == 'sirius':
    os.environ["BASE_URL"] = "http://127.0.0.1:8082/v1"
    os.environ["MODEL_NAME"] ='Qwen3.6-27B'#'ggml-org/gemma-3-1b-it-GGUF'
    os.environ["OPENAI_API_KEY"] = 'hello'
    os.environ["TRANSL_MODEL_NAME"] = 'Qwen3.6-27B'
elif use == 'cloud':
    os.environ["BASE_URL"] = "http://51.250.66.187:8080/v1"
    os.environ["MODEL_NAME"] ='Qwen3.6-27B'
    #os.environ["TRANSL_MODEL_NAME"] = 'gemma-4-12B'
    os.environ["OPENAI_API_KEY"] = 'hey'
load_dotenv(find_dotenv(usecwd=True), override=True)
import glob


%load_ext autoreload
%autoreload 2

In [10]:
df1 = pd.read_csv('new_criteria_qwen36_27b.csv')


### small fixes

In [22]:
dfo = pd.read_csv('new_criteria_ana_en.csv')
dfo.PMID = dfo.PMID.apply(lambda x: str(x.split('/')[-1]))
print(dfo.shape)

dfe = pd.read_csv('common_errors_fixed.csv')
dfe['PMID_статьи'] = dfe['PMID_статьи'].apply(lambda x: str(x).split('.')[0])

def fix(x):
    if x['PMID'] in dfe['PMID_статьи'].values:
        row = dfe[dfe['PMID_статьи'] == x['PMID']]
        row_1 =  row[row['Номер_критерия']==1]
        row_2 =  row[row['Номер_критерия']==2]
        row_3 =  row[row['Номер_критерия']==3]
        for row_i in [row_1,row_2,row_3]:
            if row_i.shape[0]:
                i = str(row_i['Номер_критерия'].values[0]).split('.')[0]
                x[f's{i}'] = row_i['Оценка_Эксперт'].values[0]
                x[f'r{i}'] = row_i['Обоснование_Эксперт'].values[0]
                print('change', x['PMID'])
        return x
    else:
        return x

dfo = dfo.apply(lambda x: fix(x), axis=1)
ps_thresh = [1,1,0]
chosen_df = dfo[(dfo['s1']>=ps_thresh[0]
                    )&(dfo['s2']>=ps_thresh[1]
                        )&(dfo['s3']>=ps_thresh[2])]
dfo['accept'] = dfo[['s1','s2','s3']].apply(lambda x: True if (x['s1']>=ps_thresh[0]
                    )&(x['s2']>=ps_thresh[1]
                        )&(x['s3']>=ps_thresh[2]) else False, axis=1)

dfo.accept.value_counts()

(67, 7)
change 25139705
change 31399568
change 26995877
change 35343863
change 35343863
change 26438251
change 26438251
change 37773318
change 37773318
change 37773318
change 33676682
change 23719267
change 23719267
change 22845480
change 24789362
change 37686423
change 41213063
change 41213063
change 36801912
change 28465216
change 28465216
change 36129915
change 36627781
change 36627781
change 34898475
change 34898475
change 28903995
change 28903995
change 23585220
change 26468446
change 23656699
change 29466887
change 40526090
change 40526090
change 38378070


accept
False    40
True     27
Name: count, dtype: int64

In [24]:
from googletrans import SyncTranslator
import re
translator = SyncTranslator()
text = (dfo.iloc[:,4:-1].values).reshape(-1)
text = [re.sub(r'\b[нН][мМ][Рр][лЛ]\b','NSCLC', re.sub(r'\b[Кк][Рр][Рр]\b','CRC',i)) for i in text]
r = translator.translate([i for i in text], dest="en")
[i.text for i in r]#.text

['The article describes the clinical case of a 19-year-old patient with malignant peritoneal mesothelioma',
 'Crizotinib has been used to treat mesothelioma',
 'The effectiveness of crizotinib for CRC has not been assessed.',
 '"16 colorectal cancer cases"',
 'The abstract states that patients received inhibitors of various signaling pathways, including ALK/ROS1. Although crizotinib is an ALK/ROS1 inhibitor, it is not explicitly named in the abstract. We cannot confirm that it was the specific drug used in patients (it could be ceritinib, alectinib, lorlatinib, etc.)',
 'The abstract provides general data for the cohort (4 patients with CRC had a response), but is not linked to a specific drug.',
 'This is a review article summarizing current advances in targeted therapy for solid tumors. It does not include original patients, models or samples',
 'Crizotinib is mentioned in the abstract, but in association with NSCLC',
 'Therapeutic efficacy for CRC has not been assessed',
 'The study

In [ ]:
dfo.iloc[:,4:-1] = np.array([i.text for i in r]).reshape(-1,3)
#dfo.to_csv('new_criteria_ana_en2.csv',index=False)

In [28]:
dfo.iloc[:,4:-1].values

array([['The article describes the clinical case of a 19-year-old patient with malignant peritoneal mesothelioma',
        'Crizotinib has been used to treat mesothelioma',
        'The effectiveness of crizotinib for CRC has not been assessed.'],
       ['"16 colorectal cancer cases"',
        'The abstract states that patients received inhibitors of various signaling pathways, including ALK/ROS1. Although crizotinib is an ALK/ROS1 inhibitor, it is not explicitly named in the abstract. We cannot confirm that it was the specific drug used in patients (it could be ceritinib, alectinib, lorlatinib, etc.)',
        'The abstract provides general data for the cohort (4 patients with CRC had a response), but is not linked to a specific drug.'],
       ['This is a review article summarizing current advances in targeted therapy for solid tumors. It does not include original patients, models or samples',
        'Crizotinib is mentioned in the abstract, but in association with NSCLC',
        

### back

In [77]:
dfo = pd.read_csv('new_criteria_ana_en2.csv')
dfo['PMID'] = dfo['PMID'].astype(str)
dfo[['s1','s2','s3']] = dfo[['s1','s2','s3']].astype(int)
dfo.head(2)

,PMID,s1,s2,s3,r1,r2,r3,accept
0,34766064,-1,-1,-1,The article describes the clinical case of a 1...,Crizotinib has been used to treat mesothelioma,The effectiveness of crizotinib for CRC has no...,False
1,32208873,1,-1,-1,"""16 colorectal cancer cases""",The abstract states that patients received inh...,The abstract provides general data for the coh...,False


In [78]:
from sklearn.model_selection import train_test_split
X = dfo.iloc[:,:1]
y = dfo.iloc[:,]
# stratify by accept/not accept. maybe instead by [1,1,0]?
X_train, X_test, y_train, y_test = train_test_split(X,y, train_size=0.7, 
                                                    random_state=42,stratify=dfo.accept)

In [123]:
import main
df_p_e = main.run_screening(pmid_list=X_train.PMID.values, 
                            fin_condition='Colorectal cancer', 
                            treatment='Crizotinib',
                            ps_thresh = [1,1,0], real=False)


INFO:root:COLLECTING PAPER TEXTS
INFO:root:2.3204500675201416
INFO:root:SCREENING PAPERS
INFO:root:0.0074841976165771484


In [124]:
df_p_e.head(2)

,PMID,s1,s2,s3,r1,r2,r3,Fin_text
0,34898475,1,1,0,The study explicitly focuses on 'colorectal ca...,The results section explicitly mentions 'S-cri...,The study identifies Crizotinib via 'in silico...,Identification of a novel immune signature for...
1,28320945,1,1,1,The study explicitly uses a 'human colorectal ...,The text states: 'Addition of the dual MET/RON...,The study evaluates therapeutic effectiveness ...,Three-dimensional culture system identifies a ...


In [80]:
y_train.head(2)

,PMID,s1,s2,s3,r1,r2,r3,accept
41,34898475,1,-1,-1,"""expression profiles and clinical characterist...",The drug is mathematically predicted to be pot...,Therapeutic efficacy has not been assessed,False
8,28320945,1,1,1,"""human colorectal cancer (CRC) cell line (HCA-...","""Addition of the dual MET/RON tyrosine kinase ...",Evaluates the therapeutic efficacy of crizotin...,True


In [128]:
errors_df = main.find_mistakes(df_p_e, y_train)
errors_df.head(1)

,PMID,num,Fin_text,s_pred,r_pred,s_expert,r_expert
3,22845480,1,Personalized medicine and pharmacogenetic biom...,1,The text explicitly mentions 'KRAS against the...,-1,"CRC is mentioned once in the list, but only in..."


In [136]:
from queries import papers_criteria as papers_criteria
error_one = errors_df.values[0]
pmid, num,abstract,model_score,model_rationale,expert_score,expert_rationale =  error_one

fin_condition='Colorectal cancer'
treatment='Crizotinib'

user_prompt = """
eligibility_criteria = "{eligibility_criteria}"
abstract = "{abstract}"
model_rationale = "{model_rationale}"
expert_rationale = "{expert_rationale}"
model_score = {model_score}
expert_score = {expert_score}
your_response = 
"""
user_prompt = user_prompt.format(eligibility_criteria = papers_criteria[num-1].format(treatment=treatment,
                                                                                      fin_condition=fin_condition),
                                 abstract=abstract,model_rationale=model_rationale,
                                 expert_rationale=expert_rationale,model_score=model_score,
                                 expert_score=expert_score)
print(user_prompt)


eligibility_criteria = "Does the study include patients, animal models, organoids, cell lines, or biological samples representing 'Colorectal cancer' as a studied disease/condition?"
abstract = "Personalized medicine and pharmacogenetic biomarkers: progress in molecular oncology testing.: : In the field of oncology, clinical molecular diagnostics and biomarker discoveries are constantly advancing as the intricate molecular mechanisms that transform a normal cell into an aberrant state in concert with the dysregulation of alternative complementary pathways are increasingly understood. Progress in biomarker technology, coupled with the companion clinical diagnostic laboratory tests, continue to advance this field, where individualized and customized treatment appropriate for each individual patient define the standard of care. Here, we discuss the current commonly used predictive pharmacogenetic biomarkers in clinical oncology molecular testing: BRAF V600E for vemurafenib in melanoma; E

## fixing types of errors

In [ ]:
df = pd.read_csv('common_errors.csv',sep=';')
chosen_cols = df.columns[~df.columns.isin(['контекст/уточнения'])]
df = df[chosen_cols].dropna(how='all')
df['Оценка_Эксперт']=df['Оценка_Эксперт'].astype(int)
df['Новая оценка эксперта'] = df[['Новая оценка эксперта','Тип_ошибки']].apply(lambda x: x['Новая оценка эксперта'] \
                                                                               if x['Тип_ошибки']!='In silico вместо фактического применения' else None,
                                                                               axis=1)
df['Оценка_Эксперт'] = df[['Оценка_Эксперт','Новая оценка эксперта']].apply(lambda x: int(x['Новая оценка эксперта']) \
                                                     if not pd.isna(x['Новая оценка эксперта']) else x['Оценка_Эксперт'],
                                                     axis=1)
df['Обоснование_Эксперт'] = df[['Обоснование_Эксперт','Новое обосновение эксперта']].apply(lambda x: x['Новое обосновение эксперта'] \
                                                     if not pd.isna(x['Новое обосновение эксперта']) else x['Обоснование_Эксперт'],
                                                     axis=1)
df['Почему_ошибка'] = df[['Почему_ошибка','Новое почему ошибка']].apply(lambda x: x['Новое почему ошибка'] \
                                                     if not pd.isna(x['Новое почему ошибка']) else x['Почему_ошибка'],
                                                     axis=1)
df['Тип_ошибки'] = df[['Новый тип ошибки','Тип_ошибки']].apply(lambda x: x['Новый тип ошибки'] if (not pd.isna(x['Новый тип ошибки'])) else x['Тип_ошибки'],
                                                         axis=1)

'In silico вместо фактического применения'
'The model mistakes a diagnostic/imaging tool for a therapeutic intervention.'

df.loc[df['Тип_ошибки'].isin(['In silico вместо фактического применения',
                         'The model mistakes a diagnostic/imaging tool for a therapeutic intervention.']),
       'Тип_ошибки'] = 'The model mistakes a diagnostic/imaging/prognostic tool for an investigated therapeutic intervention.'
df.loc[df['Тип_ошибки']=='The criteria is partially met', 'Тип_ошибки'] = 'The criteria is only partially met'
print(df.shape, df[df['Оценка_ЛЛМ']!=df['Оценка_Эксперт']].shape)
df = df[df['Оценка_ЛЛМ']!=df['Оценка_Эксперт']].iloc[:,:8]
#df.to_csv('common_errors_fixed.csv',index=False)
df.head(2)

(40, 12) (35, 12)


,PMID_статьи,Номер_критерия,Оценка_ЛЛМ,Обоснование_LLM,Оценка_Эксперт,Обоснование_Эксперт,Почему_ошибка,Тип_ошибки
0,25139705.0,1.0,1.0,The text explicitly lists 'colorectal cancer' ...,-1.0,"Это обзорная статья, обобщающая текущие достиж...",LLM приняла упоминание КРР в обзоре как фокус ...,The model confuses background keywords with th...
1,31399568.0,1.0,1.0,The text explicitly states that 'ROS1 gene rea...,-1.0,В абстракте крр упомянут как один из примеров ...,LLM интерпретировала КРР как изучаемое заболев...,The model confuses background keywords with th...


In [60]:
df['Тип_ошибки'].value_counts()

Тип_ошибки
The model confuses background keywords with the paper's focus                                            14
The model treats 'not mentioned' as 'possible'                                                           11
The model mistakes a diagnostic/imaging/prognostic tool for an investigated therapeutic intervention.     5
The criteria is only partially met                                                                        2
The model equates keyword co-occurrence with keyword interaction.                                         1
The model mistakes a proposed idea for a concept under study.                                             1
The model confuses a general topic with the potential inclusion of specific data.                         1
Name: count, dtype: int64

## translate

In [5]:
%pip uninstall -y googletrans 
%pip install googletrans-curl

Note: you may need to restart the kernel to use updated packages.


  Using cached googletrans_curl-4.1.0-py3-none-any.whl.metadata (9.1 kB)
  Using cached curl_cffi-0.16.3-cp310-abi3-win_amd64.whl.metadata (17 kB)
  Using cached cffi-2.1.1-cp310-cp310-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.11-py3-none-any.whl.metadata (8.2 kB)
Using cached googletrans_curl-4.1.0-py3-none-any.whl (21 kB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 8.5 MB/s  0:00:00

   ---------- ----------------------------- 1/4 [cffi]
   ---------- ----------------------------- 1/4 [cffi]
   ---------- ----------------------------- 1/4 [cffi]
   -------------------- ------------------- 2/4 [curl-cffi]
   -------------------- ------------------- 2/4 [curl-cffi]
   -------------------- ------------------- 2/4 [curl-cffi]
   -------------------- ------------------- 2/4 [curl-cffi]
   ------------------------------ ------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from googletrans import SyncTranslator
import re
translator = SyncTranslator()
text=['В статье описывается клинический случай 19-летнего пациента с злокачественной перитонеальной мезотелиомой',
       'Кризотиниб применялся для лечения мезотелиомы',
       'Эффективность кризотиниба для крр не оценивается','Кризотиниб упоминается в абстракте, но в связи с нмрл и транслокациями ALK']
text = [re.sub(r'\b[нН][мМ][Рр][лЛ]\b','NSCLC', re.sub(r'\b[Кк][Рр][Рр]\b','CRC',i)) for i in text]
r = translator.translate([i for i in text], dest="en")
[i.text for i in r]#.text

['The article describes the clinical case of a 19-year-old patient with malignant peritoneal mesothelioma',
 'Crizotinib has been used to treat mesothelioma',
 'The effectiveness of crizotinib for CRC has not been assessed',
 'Crizotinib is mentioned in the abstract, but in association with NSCLC and ALK translocations']

In [ ]:
from googletrans import SyncTranslator
import re
translator = SyncTranslator()
text = (dfo.iloc[:,4:].values).reshape(-1)
text = [re.sub(r'\b[нН][мМ][Рр][лЛ]\b','NSCLC', re.sub(r'\b[Кк][Рр][Рр]\b','CRC',i)) for i in text]
r = translator.translate([i for i in text], dest="en")
[i.text for i in r]#.text

['The article describes the clinical case of a 19-year-old patient with malignant peritoneal mesothelioma',
 'Crizotinib has been used to treat mesothelioma',
 'The effectiveness of crizotinib for CRC has not been assessed.',
 '"16 colorectal cancer cases"',
 'The abstract states that patients received inhibitors of various signaling pathways, including ALK/ROS1. Although crizotinib is an ALK/ROS1 inhibitor, it is not explicitly named in the abstract. We cannot confirm that it was the specific drug used in patients (it could be ceritinib, alectinib, lorlatinib, etc.)',
 'The abstract provides general data for the cohort (4 patients with CRC had a response), but is not linked to a specific drug.',
 'This is a review article summarizing current advances in targeted therapy for solid tumors. It does not include original patients, models or samples',
 'Crizotinib is mentioned in the abstract, but in association with NSCLC',
 'Therapeutic efficacy for CRC has not been assessed',
 'The study

In [ ]:
dfo.iloc[:,4:] = np.array([i.text for i in r]).reshape(-1,3)
#dfo.to_csv('/content/drive/MyDrive/new_criteria_ana_en.csv',index=False)